In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
MODEL='gpt-6-luna'

In [ ]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging
# You can choose whichever providers you like - or all Ollama

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f'OpenAI API Key exists and begins {openai_api_key[:8]}')
else:
    print('OpenAI API Key not set')
    
if anthropic_api_key:
    print(f'Anthropic API Key exists and begins {anthropic_api_key[:7]}')
else:
    print('Anthropic API Key not set')

if google_api_key:
    print(f'Google API Key exists and begins {google_api_key[:8]}')
else:
    print('Google API Key not set')

In [ ]:
# Connect to OpenAI, Anthropic and Google; comment out the Claude or Google lines if you're not using them

openai = OpenAI()

anthropic_url = 'https://api.anthropic.com/v1/'
gemini_url = 'https://generativelanguage.googleapis.com/v1beta/openai/'

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

In [ ]:
SYSTEM_PROMPT = 'You are a helpful travel assistant.  When given a destination, date range, a set of travel priorities and food preferences, you can create some itinerary options, including logistics, hotel accomodation and dining options, as well as things to do.'

In [ ]:
PRIORITIES = ['natural scenery', 'local music', 'historical sites', 'local sports', 'food', 'art museums', 'history museums']
FOOD = ['Vegetarian', 'Non-vegetarian']

def plan_trip(destination, priorities, food, start_date, end_date):
    if not destination.strip():
        yield 'Please enter a destination.'
        return
    if start_date and end_date and end_date < start_date:
        yield 'The end date must be on or after the start date.'
        return

    message = (
        f'Destination: {destination}\n'
        f'Dates: {start_date} to {end_date}\n'
        f'Travel priorities: {', '.join(priorities) if priorities else 'none specified'}\n'
        f'Food preference: {food}\n'
        'Please create some itinerary options.'
    )
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': message}]
    stream = openai.chat.completions.create(model=MODEL, reasoning_effort='none', messages=messages, stream=True)
    response = ''
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [ ]:
with gr.Blocks() as ui:
    gr.Markdown('# Trip Planner')
    with gr.Row():
        with gr.Column():
            destination = gr.Textbox(label='Destination', placeholder='e.g. Lisbon, Portugal')
            priorities = gr.Dropdown(PRIORITIES, multiselect=True, label='Travel priorities')
            food = gr.Radio(FOOD, value='Non-vegetarian', label='Food preference')
            start_date = gr.DateTime(label='Start date', include_time=False, type='string')
            end_date = gr.DateTime(label='End date', include_time=False, type='string')
            submit = gr.Button('Plan my trip', variant='primary')
        with gr.Column():
            output = gr.Markdown(label='Itinerary')
    submit.click(plan_trip, [destination, priorities, food, start_date, end_date], output)

ui.launch()